# GNN Baselines Demo

This notebook demonstrates the shared GNN link-prediction baseline pipeline on top of a saved split artifact.

It shows how to:

- load a deterministic split artifact
- configure one shared full-batch training setup
- run both `gcn` and `graphsage` with the same decoder and evaluator
- inspect the standard experiment-result payloads


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

In [ ]:
from citegnn.baselines.gnn import GNNBaselineConfig, GNNMethod, run_gnn_baseline
from citegnn.data.splits import load_split_artifact


In [ ]:
dataset_name = "cora"
seed = 42
split_path = PROJECT_ROOT / "artifacts" / "splits" / f"{dataset_name}_seed{seed}_lp.pt"

if not split_path.exists():
    raise FileNotFoundError(
        f"Missing split artifact at {split_path}. Run scripts/build_link_prediction_split.py first."
    )

split = load_split_artifact(split_path)

{
    "dataset": split.dataset_name,
    "seed": split.seed,
    "train_graph_edges": int(split.train_graph_edge_index.size(1)),
    "train_supervision_edges": int(split.train_pos_edge_index.size(1) + split.train_neg_edge_index.size(1)),
    "val_examples": int(split.val_pos_edge_index.size(1) + split.val_neg_edge_index.size(1)),
    "test_examples": int(split.test_pos_edge_index.size(1) + split.test_neg_edge_index.size(1)),
}

In [ ]:
config = GNNBaselineConfig(
    hidden_dim=32,
    num_layers=2,
    dropout=0.2,
    decoder_hidden_dim=32,
    learning_rate=0.01,
    weight_decay=5e-4,
    epochs=5,
    patience=3,
)

config.to_dict()

In [ ]:
results = [run_gnn_baseline(split, method=method, config=config).to_dict() for method in GNNMethod]
results

To switch datasets, change `dataset_name` to `"pubmed"` and rerun the notebook.

For the CLI entrypoint, use `python scripts/run_gnn_baselines.py --split-path artifacts/splits/cora_seed42_lp.pt --method all`.